In [ ]:
%cd ..
from claimbuster.adv_transformer.core.utils.flags import FLAGS


In [ ]:
import os
# display current working directory
os.getcwd()

In [ ]:
FLAGS.cs_model_dir = "/home/adamj/factcheck-podcasts/src/claimbuster/output/bba/"

In [ ]:
from claimbuster.adv_transformer.core.api.api_wrapper import ClaimSpotterAPI
claimspotter = ClaimSpotterAPI()

In [ ]:
sentence_list = [
    'Donald Trump is the 45th President of the United States',
    'I really like cheese',
    'McDonalds earns $10 billion dollars each minute'
]

In [ ]:
claimspotter.batch_sentence_query(sentence_list)

In [ ]:
import requests
podcasts = requests.get("http://127.0.0.1:8008/api/podcasts/")
podcasts = podcasts.json()

In [ ]:
# get the uuid field of each segmentation object in each segmentation_set for each transcription in transcription_set and each audioitem in audioitem_set and each podcast in podcasts
segmentation_uuids = []
for podcast in podcasts:
    for audioitem in podcast['audioitem_set']:
        for transcription in audioitem['transcription_set']:
            for segmentation in transcription['segmentation_set']:
                if segmentation['name'] == 'spaCy' and podcast['language'].startswith('en'):
                    segmentation_uuids.append(segmentation['uuid'])
len(segmentation_uuids)


In [ ]:
agent = "ClaimBuster-BBA-COREF"

for seg_uuid in segmentation_uuids:
    segmentation = requests.get(f"http://127.0.0.1:8008/api/segmentations/{seg_uuid}/")
    segmentation = segmentation.json()
    no_classifications = True
    # check if any of the utterances in segmentation["utterance_set"] have a classification made by this agent, if so, skip this segmentation
    for utterance in segmentation["utterance_set"]:
        classifications = utterance["classification_set"]
        for classification in classifications:
            if classification["agent"] == agent:
                no_classifications = False
                break
        if not no_classifications:
            break
    if not no_classifications:
        print("skipping", segmentation.get("uuid"))
        continue
    print("processing", segmentation.get("uuid"))
    sentence_list = [utt["text_coref"] if utt.get("text_coref") else utt["text"] for utt in segmentation["utterance_set"]]
    #sentence_list = [utt["text"] for utt in segmentation["utterance_set"]]
    scores = claimspotter.batch_sentence_query(sentence_list)

    for i, segment in enumerate(segmentation["utterance_set"]):
        if segment.get("text_coref"):
            requests.post(f"http://127.0.0.1:8008/api/classifications/{segment['uuid']}/", json={
                "utterance": segment["uuid"],
                "qualifier": "Checkworthiness",
                "category": "Checkworthy",
                "label": str(scores[i][1]),
                "agent": {"PROLIFIC_PID": agent},
            })

## Get Checkworthiness from Factiverse API

In [ ]:
def get_fv_checkworthiness(text, language="en"):
    query = {'lang': language, 'logging':False, 'text':text}
    response = requests.post('https://api.factiverse.no/v1/claim_detection', json=query)
    response = response.json()
    scores = [resp["score"] for resp in response["detectedClaims"]]
    score = 0 if len(scores) == 0 else sum(scores)/len(scores)
    return score


In [ ]:
for podcast in podcasts:
    language = podcast['language'][0:2] if podcast['language'] != "nb" else "no"
    print(podcast['title'], language)
    if language == "no":
        continue
    for audioitem in podcast['audioitem_set']:
        for transcription in audioitem['transcription_set']:
            for segmentation in transcription['segmentation_set']:
                if segmentation['name'] == 'spaCy':
                    seg_uuid = segmentation['uuid']
                    segmentation = requests.get(f"http://127.0.0.1:8008/api/segmentations/{seg_uuid}/")
                    segmentation = segmentation.json()

                    for segment in segmentation["utterance_set"]:
                        already_classified = False
                        if segment["classification_set"]:
                            for classification in segment["classification_set"]:
                                if classification["qualifier"] == "Checkworthiness" and classification["agent"] == "Factiverse":
                                    already_classified = True
                                    break
                        if already_classified:
                            continue
                            
                        # split the segment by spaces
                        words = segment["text"].split(" ")
                        if len(words) > 2:
                            fv_score = get_fv_checkworthiness(segment["text"], language)
                        else:
                            fv_score = 0

                        requests.post(f"http://127.0.0.1:8008/api/classifications/{segment['uuid']}/", json={
                            "utterance": segment["uuid"],
                            "qualifier": "Checkworthiness",
                            "category": "Checkworthy",
                            "label": str(fv_score),
                            "agent": "Factiverse"
                        })